## Bronze Layer: Trade Domain Archive (bronze_trade_all)


- **Purpose**: Reads temporary Parquet data from the Landing layer, updates metadata lineage (swaps `_landing_ts` for `_ingest_ts`), normalizes Batch 1 files by injecting a default `CDC_FLAG='I'` (and `CDC_DSN=NULL`) where missing, and writes to the permanent Delta archive.
- **Business Context**: PWG Pipeline - Trade Domain. This is the immutable historical archive. It preserves the exact raw data to prevent data loss from bad type casts, enabling complete pipeline replayability without ever touching the Raw ADLS zone again.
- **Execution Frequency**: Per Batch (Append-Only)
- **Inputs**:
  - `landing.trade` (Parquet)
  - `landing.tradehistory` (Parquet, Batch 1 only)
  - `landing.holdinghistory` (Parquet)
- **Outputs**:
  - `bronze.trade` (Delta, Append)
  - `bronze.tradehistory` (Delta, Append, Batch 1 only)
  - `bronze.holdinghistory` (Delta, Append)
- **Dependencies**: **MUST RUN AFTER** `landing_trade_all` (Stage 1 — Landing Zone Ingestion).

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# Configs 
dbutils.widgets.text("batch_id","1")
dbutils.widgets.text("base_path","/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg")
dbutils.widgets.text("target_schema", "charles_schwab_retailbrokerage_dev_team_lemma.bronze")
dbutils.widgets.text("domain", "TRADE")

batch_id = int(dbutils.widgets.get("batch_id"))
target_schema = dbutils.widgets.get("target_schema")
base_path = dbutils.widgets.get("base_path")
domain = dbutils.widgets.get("domain")


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
def load_bronze(file_name):
    if file_name == "trade_history" and batch_id != 1:
        print(f"Skipping: Batch-{batch_id} not applicable for {file_name}")
    else:
        # Defaults for logging in except block because run_id i am extracting from the dataframe inside try block
        carried_run_id = "UNKNOWN"
        carried_batch  = batch_id

        try:
            # Reading parquet 
            bronze_df = spark.read.format("parquet").load(f"{base_path}/Batch{batch_id}/{file_name.replace("_","")}/").filter(col("_batch") == batch_id)

            # Logging variables setup 
            carried_run_id = str(bronze_df.select("_run_id").first()[0])
            carried_batch = batch_id
            source_count = bronze_df.count()

            # Initial logging before writing to bonze delta table 
            log_pipeline_message(spark, carried_run_id, 'INFO', f'bronze_{file_name}_batch{batch_id}', 
                                 f'Starting bronze ingestion for {file_name.title()}.')
            start_pipeline_run(spark, carried_run_id, carried_batch)
            log_domain_run_status(spark, carried_run_id, carried_batch, domain, 'RUNNING')


            # Handelling holding history and trade extra cdc columns in batch 2 & 3
            if file_name in ["holdinghistory", "trade"] and batch_id == 1:
                bronze_df = bronze_df.withColumns({
                            "CDC_FLAG" : lit('I'),
                            "CDC_DSN" : lit(None).cast("string")
                            })


            # table name changes here from holdinghistory --> holdings 
            table_name = "holdings" if file_name.lower() == "holdinghistory" else file_name

#-----------------------------------------------------------------------------------------------------------------------------------------
#             # IDEMPOTENCY skip writing if same _batch id and _source_file name is already present in bronze table
#             src_tbl = bronze_df.select("_source_file").limit(1).take(1)
#             same = spark.read.table(f"{target_schema}.{table_name.lower()}")\
#                 .filter(
#                     (col("_batch") == batch_id) & (col("_source_file") == src_tbl)
#                     ).limit(1)
            
#             if same.count() > 0:
#                 print("Skipping: Same _batch id and _source_file name is already present in bronze table")
#                 return None
#             else:
# -----------------------------------------------------------------------------------------------------------------------------------------
            # Writing to bronze delta table 
            bronze_df.withColumn("_ingest_ts", current_timestamp())\
            .drop("_landing_ts")\
            .write.format("delta")\
            .mode("append").saveAsTable(f"{target_schema}.{table_name.lower()}")

            target_count = spark.read.table(f"{target_schema}.{table_name.lower()}").count()
            print(f"Successfully completed bronze ingestion for {file_name.title()} in Batch-{batch_id}. \nSource count: {source_count}, \nTarget count: {target_count}")

            # Logging success
            log_domain_run_status(spark, carried_run_id, carried_batch, domain, 'COMPLETED')
            end_pipeline_run(spark, carried_run_id, 'SUCCESS')
            log_pipeline_message(spark, carried_run_id, 'INFO', f'bronze_{file_name}_batch{batch_id}', 
                                    f'Successfully completed bronze ingestion for {file_name.title()}.')
            
            log_pipeline_recon(
                spark=spark,
                run_id=carried_run_id,
                batch_id=batch_id,
                domain=domain,
                table_name=file_name,
                source_layer="landing",
                target_layer="bronze",
                source_count=int(source_count),
                target_count=int(target_count)
            )

            log_audit_event(
                spark=spark,
                run_id=carried_run_id,
                batch=batch_id,
                layer="bronze",
                table_name=file_name,
                operation="APPEND",
                rows_affected=int(target_count)
            )

        except Exception as e:
            print(f"Failed bronze ingestion for {file_name.title()} in Batch-{batch_id} with exception {e}.") 
            log_domain_run_status(spark, carried_run_id, carried_batch, domain, f'FAILED: {file_name.title()}.')
            end_pipeline_run(spark, carried_run_id, 'FAILED')
            log_pipeline_message(spark, carried_run_id, 'ERROR', 'load_bronze', str(e))
            raise 

    return None

In [0]:
# Trade bronze
load_bronze("trade")

# Trade history bronze
load_bronze("trade_history")

# Holding history bronze
load_bronze("holdinghistory")
